# 02 - LLM + RAG test (Member C)

Tests the knowledge-enhanced LLM step (paper Section 3.4, Fig. 10) with a local model through Ollama's
OpenAI-compatible API.

1. Does switching off Qwen's reasoning ("thinking") work, and how much faster is it?
2. **(a)** The paper's two Fig. 10 cases, 3 runs each, with the paper's own context.
3. **(b)** Three made-up rules with our dataset's column names, through the full pipeline (our knowledge base + retrieval), with and without RAG.
4. **(c)** Time estimate for 300 rules x 3 runs x 2 settings.

Requirements: Ollama running with the model in `.env` (`LLM_MODEL`), `pip install -r requirements.txt`.

In [1]:
import sys, time
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(ROOT))
(ROOT / "results" / "tables").mkdir(parents=True, exist_ok=True)

from src import rag
from src import llm_diagnose as ld

pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", 250)
print({k: v for k, v in ld.SETTINGS.items() if k != "api_key"})


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.5.5 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "C:\Users\ASUS\AppData\Roaming\Python\Python312\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "C:\Users\ASUS\AppData\Roaming\Python\Python312\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "C:\Users\ASUS\AppData\Roaming\Python\Python312\site-packages\ipykernel\kernelapp.py", line 739, in start
    self.io_loop.

ImportError: 
A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.5.5 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.




A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.5.5 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "C:\Users\ASUS\AppData\Roaming\Python\Python312\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "C:\Users\ASUS\AppData\Roaming\Python\Python312\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "C:\Users\ASUS\AppData\Roaming\Python\Python312\site-packages\ipykernel\kernelapp.py", line 739, in start
    self.io_loop.

AttributeError: _ARRAY_API not found


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.5.5 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "C:\Users\ASUS\AppData\Roaming\Python\Python312\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "C:\Users\ASUS\AppData\Roaming\Python\Python312\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "C:\Users\ASUS\AppData\Roaming\Python\Python312\site-packages\ipykernel\kernelapp.py", line 739, in start
    self.io_loop.

ImportError: 
A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.5.5 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.



{'base_url': 'http://localhost:11434/v1', 'model': 'qwen3.5:4b', 'temperature': 0.7, 'top_p': 0.7, 'runs': 3, 'max_reason_words': 60}


## 1. Switching reasoning off

Same prompt (paper Case 2) sent with three settings. `reasoning chars` is the length of the separate
`reasoning` field Ollama returns; `<think> in content` checks whether thinking leaked into the answer text.

In [2]:
case2_prompt = ld.build_prompt(
    "Antecedent: Chilled water supply and return temperature difference <2 C; "
    "Consequent: Input power <100 kW; Support: 0.0491; Confidence: 0.5032",
    ["Rated chilled water supply temperature: 7 C", "Rated chilled water return temperature: 12 C",
     "The temperature difference between the refrigeration supply and return water is less than 2 C, "
     "and the heat exchange is insufficient"])

def raw_call(messages, **extra):
    t0 = time.perf_counter()
    r = ld._get_client().chat.completions.create(
        model=ld.SETTINGS["model"], messages=messages, temperature=0.7, top_p=0.7,
        response_format={"type": "json_object"}, **extra)
    msg = r.choices[0].message
    reasoning = getattr(msg, "reasoning", None) or (msg.model_extra or {}).get("reasoning") or ""
    return {"latency_s": round(time.perf_counter() - t0, 1), "completion_tokens": r.usage.completion_tokens,
            "reasoning chars": len(reasoning), "<think> in content": "<think>" in (msg.content or "")}

no_think_msgs = [case2_prompt[0], {"role": "user", "content": case2_prompt[1]["content"] + " /no_think"}]
think_test = pd.DataFrame([
    {"method": "default (thinking on)", **raw_call(case2_prompt)},
    {"method": 'extra_body={"think": False}', **raw_call(case2_prompt, extra_body={"think": False})},
    {"method": "/no_think in user message", **raw_call(no_think_msgs)},
    {"method": 'reasoning_effort="none" (used)', **raw_call(case2_prompt, **ld.NO_THINKING)},
])
think_test

,method,latency_s,completion_tokens,reasoning chars,<think> in content
0,default (thinking on),88.8,3887,16325,False
1,"extra_body={""think"": False}",76.0,3887,16453,False
2,/no_think in user message,75.0,3883,16368,False
3,"reasoning_effort=""none"" (used)",1.8,74,0,False


## (a) Paper Fig. 10 cases, 3 runs each

Rules and context text exactly as in the paper. Expected: Case 1 validity false / label 0,
Case 2 validity true / label 2.

In [3]:
cases = [
    {"case": 1,
     "rule": {"antecedent": ("Cooling capacity=[1500,2300] kW",), "consequent": ("System COP=[4,6]",),
              "support": 0.2503, "confidence": 0.927, "lift": None},
     "context": ["Rated cooling capacity: 2800 kW", "Rated COP: 5.92",
                 "System COP is in the normal range of 4-6",
                 "Cooling capacity is 1500-2300 kW, normal load range"],
     "expected": (False, 0)},
    {"case": 2,
     "rule": {"antecedent": ("Chilled water supply and return temperature difference <2 C",),
              "consequent": ("Input power <100 kW",), "support": 0.0491, "confidence": 0.5032, "lift": None},
     "context": ["Rated chilled water supply temperature: 7 C", "Rated chilled water return temperature: 12 C",
                 "The temperature difference between the refrigeration supply and return water is less "
                 "than 2 C, and the heat exchange is insufficient"],
     "expected": (True, 2)},
]

rows = []
for c in cases:
    prompt = ld.build_prompt(ld.rule_to_text(c["rule"]), c["context"])
    for run in range(1, ld.SETTINGS["runs"] + 1):
        r = ld.classify_rule(prompt)
        rows.append({"case": c["case"], "run": run, "validity": r["validity"], "label": r["label"],
                     "reason": r["reason"] or r["error"],
                     "matches paper": (r["validity"], r["label"]) == c["expected"],
                     "latency_s": round(r["latency_s"], 2), "tokens": r["total_tokens"], "failed": r["failed"]})
paper_cases = pd.DataFrame(rows)
paper_cases.to_csv(ROOT / "results/tables/llm_paper_cases.csv", index=False)
print("Prompt sent for Case 2:\n")
print(ld.build_prompt(ld.rule_to_text(cases[1]["rule"]), cases[1]["context"])[1]["content"])
paper_cases

Prompt sent for Case 2:

[Rule]
Antecedent: Chilled water supply and return temperature difference <2 C; Consequent: Input power <100 kW; Support: 0.0491; Confidence: 0.5032

[Context]
- Rated chilled water supply temperature: 7 C
- Rated chilled water return temperature: 12 C
- The temperature difference between the refrigeration supply and return water is less than 2 C, and the heat exchange is insufficient

[User]
Please reply in JSON with three fields: 1. "validity": true or false; 2. "label": one of {"0 Invalid rules", "1 Design anomaly", "2 Operational anomaly", "3 Performance degradation"}; 3. "reason": concise (<=60 words) explanation.


,case,run,validity,label,reason,matches paper,latency_s,tokens,failed
0,1,1,True,0,"The rule describes a normal operational state where the system's cooling capacity and COP fall within their expected ranges. It does not indicate any design flaws, performance degradation, or anomalous behavior.",False,1.53,285,False
1,1,2,True,0,The rule is valid as it correctly identifies the expected normal operating range for cooling capacity and COP based on system ratings.,False,1.07,271,False
2,1,3,True,3,"The rule indicates high confidence that systems with rated capacity >2800 kW exhibit lower operational COP (<4). This suggests the large unit is underperforming relative to its design rating of 5.92, likely due to part-load inefficiencies or control issues.",False,1.70,303,False
3,2,1,True,2,"A temperature difference below 2°C indicates insufficient heat exchange in the chilled water loop. This typically stems from low refrigerant charge or flow issues, representing an operational fault rather than a design defect.",True,1.44,278,False
4,2,2,True,2,"A supply/return temperature difference below 2°C indicates insufficient heat exchange in the chilled water loop. This suggests a fault such as low refrigerant charge, compressor issues, or flow restriction, rather than normal operation or design parameters.",True,1.52,285,False
5,2,3,True,2,"A supply-return temperature difference under 2°C indicates insufficient heat exchange due to low water flow or chiller malfunction. This operational state correlates with high input power relative to cooling output, confirming an active system fault rather than a design limitation.",True,1.53,286,False


In [4]:
paper_cases.groupby("case")["matches paper"].agg(["sum", "count"]).rename(columns={"sum": "matches", "count": "runs"})

,matches,runs
case,,
1,0,3
2,3,3


## (b) Made-up rules with our column names: full pipeline

These rules are invented for testing (not mined); "our expectation" is our own judgement from the
control sequence, not ground truth.

| # | Rule | Our expectation |
|---|---|---|
| 1 | high outdoor dry-bulb -> low chilled-water set-point | 0: normal set-point reset |
| 2 | tower fan high + low wet-bulb -> tower leaving water high | 3: high approach, degradation |
| 3 | low load + secondary pump speed high -> low differential pressure | 2: operational (pump/DP control) |

In [5]:
test_rules = [
    {"antecedent": ("OA_TEMP=high",), "consequent": ("CWL_PRI_SW_TEMPSPT=low",),
     "support": 0.21, "confidence": 0.93, "lift": 3.1},
    {"antecedent": ("CT_FAN_SPD_1=high", "OA_TEMP_WB=low"), "consequent": ("CT_SW_TEMP_1=high",),
     "support": 0.04, "confidence": 0.71, "lift": 2.4},
    {"antecedent": ("CWL_SEC_LOAD=low", "CWL_SEC_PM_SPD_1=high"), "consequent": ("CWL_SEC_DP=low",),
     "support": 0.03, "confidence": 0.65, "lift": 1.9},
]
expected_label = {0: 0, 1: 3, 2: 2}

for i, rule in enumerate(test_rules):
    text = ld.rule_to_text(rule)
    print(f"Rule {i}: {text}")
    for line in rag.describe_variables(text):
        print(f"   def: {line}")
    for c in rag.retrieve(text):
        print(f"   {c['score']:.3f} [{c['source']}#{c['id']}] {c['text']}")
    print()

Rule 0: Antecedent: OA_TEMP=high; Consequent: CWL_PRI_SW_TEMPSPT=low; Support: 0.2100; Confidence: 0.9300; Lift: 3.10
   def: OA_TEMP = outdoor air dry-bulb temperature (degF)
   def: CWL_PRI_SW_TEMPSPT = chilled water supply temperature set-point (degF)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

   0.588 [02_control_sequence.md#11] Chilled-water supply set-point reset: the chiller leaving-water set-point (CWL_PRI_SW_TEMPSPT) is reset linearly from outdoor dry-bulb temperature, 52 degF when outdoor air is 60 degF or colder down to 42 degF when it is 80 degF or warmer, updated hourly. A low chilled-water set-point in hot weather is therefore expected.
   0.563 [02_control_sequence.md#13] Condenser water set-point: the tower leaving-water set-point (CT_SW_TEMPSPT) equals outdoor wet-bulb temperature + 8 degF approach, with a lower limit of 34 degF. All running tower fans run at the same speed, controlled by a feedback loop to track this set-point.
   0.554 [01_plant_design.md#5] Units in the data: temperatures in degF, water flows in GPM, power in kW, the secondary-loop cooling load (CWL_SEC_LOAD) in W, differential pressure in inH2O; control signals, speeds and valve position are fractions 0-1; status points are 0 = off, 1 = on.

Rule 1: Antecedent: CT_FAN_SPD_1=high AND OA_TEMP

In [6]:
df_rag = ld.classify_all(test_rules, use_rag=True, out_name="llm_test_rules_rag")
df_norag = ld.classify_all(test_rules, use_rag=False, out_name="llm_test_rules_norag")

both = pd.concat([df_rag, df_norag])
both["rule"] = both["antecedent"] + " -> " + both["consequent"]
both["our expectation"] = both["rule_id"].map(expected_label)
both["matches"] = both["label"] == both["our expectation"]
both["latency_s"] = both["latency_s"].round(2)
both[["rule_id", "use_rag", "run", "validity", "label", "our expectation", "matches", "reason",
      "latency_s", "total_tokens", "failed"]].sort_values(["rule_id", "use_rag", "run"])

rules:   0%|          | 0/3 [00:00<?, ?it/s]

saved 9 rows to results\tables\llm_test_rules_rag.csv  (failed: 0)


rules:   0%|          | 0/3 [00:00<?, ?it/s]

saved 9 rows to results\tables\llm_test_rules_norag.csv  (failed: 0)


,rule_id,use_rag,run,validity,label,our expectation,matches,reason,latency_s,total_tokens,failed
0,0,False,1,True,2,0,False,"High outdoor temperature typically increases cooling load, requiring higher compressor output. A low setpoint with high ambient conditions suggests incorrect control logic or sensor malfunction, indicating an operational issue rather than a design flaw.",9.42,234,False
1,0,False,2,True,1,0,False,"High outdoor temperature typically increases cooling load, requiring higher compressor temperatures. A low setpoint during high heat suggests a design flaw or control logic error, indicating an anomaly rather than normal operation.",8.78,232,False
2,0,False,3,True,2,0,False,"High outdoor temperature with low cooling setpoint indicates an active cooling cycle running at minimum capacity to manage heat gain, which is a typical operational response rather than a design flaw or performance degradation.",8.77,232,False
0,0,True,1,True,0,0,True,"The rule is valid because it accurately reflects the designed control strategy. As outdoor temperature rises, the chilled water set-point resets downward to maintain efficiency and capacity, making 'high' OA temp correlate strongly with 'low' CWL set-points.",13.64,503,False
1,0,True,2,True,2,0,False,"The rule indicates a high outdoor temperature triggers a low chilled water set-point, which contradicts standard cooling logic where set-points rise in heat to prevent overcooling. This suggests an unintended operational behavior or control system error rather than design intent.",10.30,503,False
2,0,True,3,True,2,0,False,"The rule indicates high outdoor temperatures causing a low chilled water set-point, which contradicts standard cooling logic where hot weather requires higher supply temperatures to meet demand. This suggests an operational error or control malfunction.",9.22,495,False
3,1,False,1,True,2,3,False,High fan speed combined with low outdoor dry-bulb temperature causing high indoor coil temperature indicates a control failure or sensor drift. This operational mismatch suggests improper cooling performance rather than design flaws.,9.14,242,False
4,1,False,2,True,1,3,False,"The rule indicates high fan speed correlates with low outlet air temperature and high sensor reading, suggesting a control mismatch or sensor drift where cooling intent conflicts with measured output.",8.11,237,False
5,1,False,3,True,1,3,False,"High fan speed combined with low outdoor dry-bulb temperature typically indicates a control strategy error or sensor fault causing unnecessary cooling load, rather than normal operation or performance degradation.",8.38,239,False
3,1,True,1,True,3,3,True,"High fan speed combined with low wet-bulb temperature indicates the system is trying to reject heat aggressively but failing to reduce supply temperature. This signifies fouling or scaling reducing heat transfer efficiency, causing performance degradation.",10.26,468,False


In [7]:
summary = both.groupby(["use_rag", "rule_id"]).agg(
    labels=("label", lambda s: list(s)), matches=("matches", "sum"), mean_latency_s=("latency_s", "mean"),
    mean_tokens=("total_tokens", "mean")).round(2)
summary

labels  matches  mean_latency_s  mean_tokens
use_rag rule_id                                                 
False   0        [2, 1, 2]        0            8.99       232.67
        1        [2, 1, 1]        0            8.54       239.33
        2        [2, 2, 2]        3            8.46       238.67
True    0        [0, 2, 2]        1           11.05       500.33
        1        [3, 2, 2]        1           10.43       473.00
        2        [2, 2, 2]        3            9.38       485.33

## (c) Time estimate: 300 rules x 3 runs x 2 settings

From the latencies measured above (RAG prompts are longer, so the two settings are timed separately).
Assumes calls run one after another on this machine.

In [8]:
lat_rag = df_rag["latency_s"].mean()
lat_norag = df_norag["latency_s"].mean()
n_rules, n_runs = 300, ld.SETTINGS["runs"]
calls = n_rules * n_runs
hours = calls * (lat_rag + lat_norag) / 3600
estimate = pd.DataFrame([
    {"setting": "with RAG", "calls": calls, "mean latency (s)": round(lat_rag, 2), "hours": round(calls * lat_rag / 3600, 2)},
    {"setting": "without RAG", "calls": calls, "mean latency (s)": round(lat_norag, 2), "hours": round(calls * lat_norag / 3600, 2)},
    {"setting": "total", "calls": 2 * calls, "mean latency (s)": None, "hours": round(hours, 2)},
])
estimate.to_csv(ROOT / "results/tables/llm_time_estimate.csv", index=False)
estimate

,setting,calls,mean latency (s),hours
0,with RAG,900,10.29,2.57
1,without RAG,900,8.66,2.17
2,total,1800,NaN,4.74


## Notes (run of 22 Nov 2026, qwen3.5:4b on Ollama 0.40.16)

- **Reasoning off works** with `reasoning_effort="none"` only (1.8 s, 74 tokens, vs. 75-89 s and ~3,900 tokens with
  reasoning on). `extra_body={"think": False}` and `/no_think` were ignored by the OpenAI-compatible endpoint.
  Thinking text came back in a separate `reasoning` field, never as `<think>` in the answer (stripped anyway).
- **Paper cases:** Case 2 matched the paper in 3/3 runs. Case 1 matched 0/3: the label was 0 in 2 runs, but the model
  always returned `validity: true`, apparently reading "validity" as "the rule is factually correct" rather than
  "the rule is diagnostically useful". This is a weakness of the paper's prompt wording with a small model.
- **Test rules:** with RAG the model got the expected label in 5/9 runs, without RAG in 3/9. The retrieved context
  helped on rule 0 (set-point reset) and rule 1 (tower approach) only in 1 of 3 runs each; a 4B model at
  temperature 0.7 is inconsistent, so voting over runs (Member D's improvement) should help.
- **Latency varies with machine load:** about 1.5 s per call early in this run and 8-12 s later (also re-checked
  outside the notebook with tiny prompts), so the slowdown is the machine, not the prompt length.
  Time estimate for 1,800 calls: about **0.75 h** at 1.5 s/call up to the **4.7 h** measured above. Run the full
  experiment when nothing else is using the GPU.
- The `ImportError ... NumPy 1.x` messages in the setup cell come from optional pandas add-ons (pyarrow, numexpr,
  bottleneck) in the Anaconda environment used to run this notebook; they are harmless warnings.
- A local model costs nothing per token; the limit is time on this machine.
- The knowledge base (`knowledge_base/`) contains general HVAC knowledge and the plant description only,
  no dataset-specific answers.